# 02. RDD (Resilient Distributed Datasets) Fundamentals

In this notebook, you will learn the low-level foundation of Apache Spark: **RDDs**.

### Topics Covered:
1. **What is an RDD and why is it called that?**
2. **Creating RDDs using `sc.parallelize()` and Partitions**
3. **Narrow vs. Wide Transformations**
4. **Essential RDD Transformations (`map`, `filter`, `flatMap`, `reduceByKey`)**
5. **Essential RDD Actions (`collect`, `count`, `take`, `reduce`)**
6. **Classic Big Data Example: Word Count**
7. **When to use RDDs vs DataFrames in modern PySpark**
8. **Stopping the SparkSession to free memory**


---
## 1. What is an RDD?

**RDD** stands for **Resilient Distributed Dataset**:
* **Resilient**: Fault-tolerant. If any worker node crashes during a calculation, Spark can reconstruct lost data automatically using its **Lineage Graph** (the history of transformations).
* **Distributed**: The data is partitioned across multiple worker nodes in the cluster so it can be processed in parallel.
* **Dataset**: A collection of values or records (strings, numbers, objects, tuples).

### 3 Key Properties of RDDs:
1. **Immutable**: Once created, an RDD cannot be modified. Any transformation creates a *new* RDD.
2. **Lazy**: Transformations are not executed until an Action is called.
3. **Partitioned**: Data is split into smaller logical chunks called **Partitions**. Each partition is processed by a separate thread or core.


---
## 2. Initializing SparkSession & SparkContext

To work with RDDs, we use the `SparkContext`, which is accessible from `spark.sparkContext`.


In [1]:


from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("02_RDD_Fundamentals") \
    .master("local[2]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

# We get the SparkContext from the session
sc = spark.sparkContext
sc.setLogLevel("WARN")

print(f"SparkContext is ready! Master: {sc.master}")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/05 18:00:53 WARN Utils: Your hostname, Tiger, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/05 18:00:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/05 18:00:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkContext is ready! Master: local[2]


---
## 3. Creating an RDD

You can create an RDD from an in-memory Python collection using `sc.parallelize()`.
You can also specify the number of **partitions**.


In [2]:
# Python list of numbers
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# Create RDD with 3 partitions
numbers_rdd = sc.parallelize(numbers, numSlices=3)

print("Type of RDD:", type(numbers_rdd))
print("Number of partitions:", numbers_rdd.getNumPartitions())
# Glom collects each partition as a separate list so we can see data distribution
print("Data split across partitions:", numbers_rdd.glom().collect())


Type of RDD: <class 'pyspark.core.rdd.RDD'>
Number of partitions: 3


Data split across partitions: [[1, 2, 3], [4, 5, 6], [7, 8, 9, 10]]


---
## 4. Narrow vs Wide Transformations

Understanding this concept is crucial for writing fast Spark code:

### A. Narrow Transformations (Fast, No Shuffle)
* Each partition of the parent RDD is used by at most one partition of the child RDD.
* No data needs to travel across the network between machines.
* Examples: `map()`, `filter()`, `flatMap()`.

### B. Wide Transformations (Slower, Requires Data Shuffling)
* Multiple child partitions depend on data from multiple parent partitions.
* Data must be reorganized and sent across machines over the network (called a **Shuffle**).
* Examples: `groupByKey()`, `reduceByKey()`, `distinct()`, `sortBy()`.


---
## 5. Narrow Transformations in Action: `map`, `filter`, and `flatMap`


In [3]:
# A. map(): Applies a function to EVERY element (1-to-1 mapping)
# Let's square every number: x -> x * x
squared_rdd = numbers_rdd.map(lambda x: x * x)
print("Squared numbers (map):", squared_rdd.collect())


Squared numbers (map): [1, 4, 9, 16, 25, 36, 49, 64, 81, 100]


In [4]:
# B. filter(): Keeps only elements that satisfy a condition
# Let's keep only even numbers: x % 2 == 0
evens_rdd = numbers_rdd.filter(lambda x: x % 2 == 0)
print("Even numbers (filter):", evens_rdd.collect())


Even numbers (filter): [2, 4, 6, 8, 10]


In [6]:
# C. flatMap(): Similar to map, but flattens lists into individual elements (1-to-many)
sentences = ["Hello PySpark", "Big Data is awesome", "RDD fundamentals"]
sentences_rdd = sc.parallelize(sentences)

# Using map: gives list of lists
mapped_words = sentences_rdd.map(lambda s: s.split(" ")).collect()
print("Using map (nested lists):", mapped_words)

# Using flatMap: flattens into a single list of all words
flat_words = sentences_rdd.flatMap(lambda s: s.split(" ")).collect()
print("Using flatMap (flattened list):", flat_words)


Using map (nested lists): [['Hello', 'PySpark'], ['Big', 'Data', 'is', 'awesome'], ['RDD', 'fundamentals']]
Using flatMap (flattened list): ['Hello', 'PySpark', 'Big', 'Data', 'is', 'awesome', 'RDD', 'fundamentals']


---
## 6. Wide Transformations: Key-Value RDDs & `reduceByKey`

In Big Data, datasets often consist of key-value pairs (e.g. `(category, amount)` or `(word, 1)`).

### Why `reduceByKey` is better than `groupByKey`:
* `groupByKey()` transfers all values for a key across the network before aggregating. This causes massive memory spikes and slow network transfers.
* `reduceByKey()` combines values **locally on each worker first** (map-side combine) before sending the reduced sums across the network. This dramatically reduces network traffic!


In [7]:
# List of sales: (Department, Revenue)
sales_data = [
    ("Electronics", 1200),
    ("Clothing", 450),
    ("Electronics", 800),
    ("Groceries", 300),
    ("Clothing", 150),
    ("Electronics", 600)
]

sales_rdd = sc.parallelize(sales_data)

# reduceByKey: takes two values for the same key and merges them
total_sales_per_dept = sales_rdd.reduceByKey(lambda x, y: x + y)

print("Total revenue per department (reduceByKey):")
for dept, total in total_sales_per_dept.collect():
    print(f"  - {dept}: ${total}")


Total revenue per department (reduceByKey):
  - Clothing: $600
  - Groceries: $300
  - Electronics: $2600


---
## 7. RDD Actions (Triggering Execution)

Transformations are lazy. Actions force Spark to actually compute the result.


In [8]:
# 1. count(): Total number of elements
print("Count of sales records:", sales_rdd.count())

# 2. first(): Returns the first element
print("First record:", sales_rdd.first())

# 3. take(n): Returns the first n elements safely (avoids collecting millions of rows)
print("First 2 records:", sales_rdd.take(2))

# 4. reduce(): Aggregates all elements into a single scalar value
numbers_to_sum = sc.parallelize([10, 20, 30, 40, 50])
total_sum = numbers_to_sum.reduce(lambda a, b: a + b)
print("Total sum via reduce action:", total_sum)

# WARNING about collect():
# .collect() pulls ALL data from the cluster to the Driver machine.
# If your dataset is 100 GB and your laptop RAM is 16 GB, collect() WILL CRASH your program!
# Always use .take(n) to preview data in production.
print("Safely previewing 3 items:", sales_rdd.take(3))


Count of sales records: 6
First record: ('Electronics', 1200)
First 2 records: [('Electronics', 1200), ('Clothing', 450)]
Total sum via reduce action: 150
Safely previewing 3 items: [('Electronics', 1200), ('Clothing', 450), ('Electronics', 800)]


---
## 8. Classic Big Data Project: Word Count with RDD

The "Hello World" of distributed computing is counting the frequency of every word in a document.


In [9]:
# Sample text document lines
text_data = [
    "Apache Spark is a unified analytics engine for big data processing",
    "Spark provides high-level APIs in Python Java Scala and R",
    "PySpark makes it easy to process big data with Python",
    "Spark is fast because of in-memory processing"
]

# Step 1: Create RDD
text_rdd = sc.parallelize(text_data)

# Step 2: Split sentences into words and convert to lowercase (flatMap)
words = text_rdd.flatMap(lambda line: line.lower().split(" "))

# Step 3: Map each word to a pair (word, 1)
word_pairs = words.map(lambda word: (word, 1))

# Step 4: Reduce by key to sum the counts
word_counts = word_pairs.reduceByKey(lambda count1, count2: count1 + count2)

# Step 5: Sort by word count in descending order
sorted_word_counts = word_counts.sortBy(lambda pair: pair[1], ascending=False)

print("Top 10 Most Frequent Words:")
for word, count in sorted_word_counts.take(10):
    print(f"  '{word}': {count}")


Top 10 Most Frequent Words:
  'spark': 3
  'big': 2
  'python': 2
  'is': 2
  'data': 2
  'processing': 2
  'apache': 1
  'analytics': 1
  'for': 1
  'java': 1


---
## 9. RDDs vs DataFrames: Which Should You Use?

* **Use DataFrames (95% of the time):**
  * Modern PySpark uses **DataFrames** because Spark's **Catalyst Optimizer** automatically optimizes SQL queries and operations.
  * DataFrames are much faster than raw Python RDDs because they avoid Python-JVM serialization overhead.
* **Use RDDs (5% of the time):**
  * When working with unstructured data (raw binary files, complex custom text streams).
  * When you need low-level control over partitioning and physical data placement.

---
## ⚠️ CRITICAL STEP: Stopping the SparkSession
Always release RAM and JVM background processes before moving to the next notebook.


In [10]:
# Stop SparkSession to free memory
spark.stop()
print("SparkSession stopped successfully! RAM released.")


SparkSession stopped successfully! RAM released.
